In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

import json
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences

# IMPORTANT:
# This imports the custom Transformer classes so Keras can
# locate the classes stored inside transformer_summarization.keras.
import transformer_summarization_custom

print("TensorFlow:", tf.__version__)
print("Custom Transformer classes loaded.")


TensorFlow: 2.21.0
Custom Transformer classes loaded.


In [2]:
# ============================================================
# 2. LOAD SAVED MODEL
# ============================================================

# These files are created from the TRAINING notebook
MODEL_PATH = "Model/transformer_summarization.keras"
TEXT_TOKENIZER_PATH = "Model/text_tokenizer.json"
SUMMARY_TOKENIZER_PATH = "Model/summary_tokenizer.json"

transformer = load_model(
    MODEL_PATH,
    compile=False
)

print("Model loaded successfully.")


Model loaded successfully.


In [3]:
# ============================================================
# 3. LOAD TOKENIZERS
# ============================================================

with open(TEXT_TOKENIZER_PATH, "r", encoding="utf-8") as f:
    text_tokenizer = tokenizer_from_json(f.read())

with open(SUMMARY_TOKENIZER_PATH, "r", encoding="utf-8") as f:
    summary_tokenizer = tokenizer_from_json(f.read())

print("Tokenizers loaded successfully.")

Tokenizers loaded successfully.


In [4]:
# ============================================================
# 4. LOAD MODEL CONFIG
# ============================================================

with open("Model/summarization_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

max_input_seq_length = config["max_input_seq_length"]
max_output_seq_length = config["max_output_seq_length"]

sos_id = summary_tokenizer.word_index["<sos>"]
eos_id = summary_tokenizer.word_index["<eos>"]

print("Max input length:", max_input_seq_length)
print("Max output length:", max_output_seq_length)

Max input length: 45
Max output length: 18


In [5]:
# ============================================================
# 5. SUMMARIZE NEW TEXT
# ============================================================

def summarize(text, max_length=None):

    if max_length is None:
        max_length = max_output_seq_length - 1

    encoder_ids = text_tokenizer.texts_to_sequences([text])

    encoder_ids = pad_sequences(
        encoder_ids,
        maxlen=max_input_seq_length,
        padding="post",
        truncating="post"
    )

    decoder_ids = [sos_id]

    for _ in range(max_length):

        decoder_input = pad_sequences(
            [decoder_ids],
            maxlen=max_output_seq_length - 1,
            padding="post",
            truncating="post"
        )

        logits = transformer(
            [
                tf.constant(encoder_ids, dtype=tf.int32),
                tf.constant(decoder_input, dtype=tf.int32)
            ],
            training=False
        )

        step = len(decoder_ids) - 1

        next_id = int(
            tf.argmax(logits[0, step]).numpy()
        )

        if next_id == eos_id or next_id == 0:
            break

        decoder_ids.append(next_id)

    index_to_word = {
        idx: word
        for word, idx in summary_tokenizer.word_index.items()
    }

    words = []

    for idx in decoder_ids[1:]:

        if idx == eos_id:
            break

        words.append(
            index_to_word.get(idx, "<unk>")
        )

    return " ".join(words)

In [8]:
# ============================================================
# 6. ASK THE MODEL
# ============================================================

new_text = "Artificial intelligence is transforming healthcare by helping doctors analyze medical images, identify possible diseases, organize patient records, and support clinical decisions. AI systems can process large amounts of medical information quickly, but doctors still need to review important results before making decisions."

print("NEW TEXT:")
print(new_text)

print("\nGENERATED SUMMARY:")
print(summarize(new_text))

NEW TEXT:
Artificial intelligence is transforming healthcare by helping doctors analyze medical images, identify possible diseases, organize patient records, and support clinical decisions. AI systems can process large amounts of medical information quickly, but doctors still need to review important results before making decisions.

GENERATED SUMMARY:
ai is transforming healthcare by analyzing medical information and supporting doctors, while human review remains important.


In [7]:
# ============================================================
# 7. ASK ANOTHER TEXT
# ============================================================

new_text = """A company has introduced an AI assistant that can analyze
documents, answer questions, and create short summaries. The system
uses a Transformer model to understand relationships between words
and generate useful responses. The company plans to add support for
more languages in the future."""

print("NEW TEXT:")
print(new_text)

print("\nGENERATED SUMMARY:")
print(summarize(new_text))

NEW TEXT:
A company has introduced an AI assistant that can analyze
documents, answer questions, and create short summaries. The system
uses a Transformer model to understand relationships between words
and generate useful responses. The company plans to add support for
more languages in the future.

GENERATED SUMMARY:
nlp enables computers to process human language for tasks such as translation, chatbots, and summarization.
